# CascadeFlow VLM Routing Experiment

This notebook is a thin orchestrator around `code_base/cascadeflow/cascadeflow_experiment.py`. Configure the experiment parameters, let the helper functions download/resolve images on demand, and run the routing pipeline end-to-end without touching any async code.

## Notebook flow
1. Import the helper module and set notebook defaults.
2. Configure dataset paths, cascade model specs, and generation knobs.
3. (Optional) Inspect a prepared sample and display the image retrieved from Cauldron.
4. Run the CascadeFlow experiment and capture the routing telemetry.
5. Summarize the model mix/costs/latency and persist the detailed logs to disk.

In [28]:
# %% [markdown]
# # VLM Router — Dataset, Images & CascadeFlow Experiment
#
# This notebook does:
# 1. **Config setup** (paths, models, experiment knobs)
# 2. **Load router dataset + basic EDA**
# 3. **Build Cauldron image lookup & resolve `resolved_image_path`**
# 4. **Display sample images + prompts**
# 5. **Configure CascadeFlow & vLLM models**
# 6. **Smoke-test a few samples**
# 7. **Run full experiment & save results**


## Setup

In [29]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
# %%
from __future__ import annotations

import os
import sys
import asyncio
import logging
import base64
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional

import pandas as pd
from PIL import Image
from IPython.display import display

try:
    from tqdm.auto import tqdm
except Exception:  # simple fallback
    def tqdm(x, **kwargs):
        return x

from cascadeflow import CascadeAgent, ModelConfig  # pip install cascadeflow[all]

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("vlm_router")
logger.setLevel(logging.INFO)


In [41]:
# %% [markdown]
# ## 1. Paths & global config
#
# Adjust these if your directory layout is different.

# %%
# Detect project root (you can hard-code if needed)
PROJECT_ROOT = Path.cwd().resolve().parent.parent
print("PROJECT_ROOT:", PROJECT_ROOT)

# Router dataset (with sample_id, prompt_formatted, etc.)
DATASET_PATH = PROJECT_ROOT / "dataset/final_dataset/router_pivot_dataset_train.parquet"

# Cauldron metadata (for mapping dataset/index → image path)
CAULDRON_LOOKUP_PATH = PROJECT_ROOT / "dataset/which_vlm_data/processed/cauldron_poc_multi.parquet"

# Where images are stored/cached
IMAGE_ROOT = PROJECT_ROOT / "dataset/which_vlm_data/images/cauldron"
IMAGE_ROOT.mkdir(parents=True, exist_ok=True)

# Where experiment results will be written
OUTPUT_DIR = PROJECT_ROOT / "dataset/which_vlm_data/results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("DATASET_PATH:", DATASET_PATH)
print("CAULDRON_LOOKUP_PATH:", CAULDRON_LOOKUP_PATH)
print("IMAGE_ROOT:", IMAGE_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)


PROJECT_ROOT: /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router
DATASET_PATH: /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/dataset/final_dataset/router_pivot_dataset_train.parquet
CAULDRON_LOOKUP_PATH: /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/dataset/which_vlm_data/processed/cauldron_poc_multi.parquet
IMAGE_ROOT: /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/dataset/which_vlm_data/images/cauldron
OUTPUT_DIR: /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/dataset/which_vlm_data/results


In [42]:
# %% [markdown]
# ## 2. Optional: Cauldron fetch function
#
# If your `dataset_builder.check_data_utils` module is present, we'll use
# `fetch_cauldron_image`. If not, you'll still be able to use existing images.

# %%
fetch_cauldron_image = None

WHICH_VLM_SRC = PROJECT_ROOT / "code_base" / "which_vlm"
if WHICH_VLM_SRC.exists():
    which_vlm_str = str(WHICH_VLM_SRC)
    if which_vlm_str not in sys.path:
        sys.path.append(which_vlm_str)

    try:
        from dataset_builder.check_data_utils import fetch_cauldron_image  # type: ignore
        print(" Imported fetch_cauldron_image from dataset_builder.check_data_utils")
    except Exception as e:
        print("️ Could not import fetch_cauldron_image:", e)
else:
    print("️ WHICH_VLM_SRC not found, skipping fetch_cauldron_image import.")


✅ Imported fetch_cauldron_image from dataset_builder.check_data_utils


In [43]:
# %% [markdown]
# ## 3. Image helpers (small, debuggable pieces)
#
# We'll build everything out of small functions so it's easy to debug.

# %%
def debug_row_metadata(row: pd.Series, extra: Optional[str] = None) -> None:
    """Print key fields for a single row to help debugging."""
    header = f"[DEBUG ROW] sample_id={row.get('sample_id')}"
    if extra:
        header += f" | {extra}"
    print(header)

    keys = [
        "source_dataset",
        "source_config",
        "source_index",
        "cauldron_lookup_key",
        "image_path",
        "image_path_absolute",
        "image_cache_root",
        "cauldron_image_asset",
    ]
    for k in keys:
        if k in row.index:
            print(f"  {k}: {row.get(k)}")


In [44]:
# %%
def build_lookup_key(dataset_name: str, source_index: int | str) -> str:
    """Consistently build 'dataset:index' keys."""
    return f"{dataset_name}:{int(source_index)}"


In [45]:
# %%
def candidate_paths_from_row(row: pd.Series, project_root: Path) -> List[Path]:
    """
    Collect candidate file paths from the row itself.
    Pure function, easy to unit-test.
    """
    candidates: List[Path] = []

    def _append_str_path(value: Any) -> None:
        if isinstance(value, str) and value:
            p = Path(value)
            if not p.is_absolute():
                p = (project_root / p).resolve()
            candidates.append(p)

    # Direct paths
    _append_str_path(row.get("image_path"))
    _append_str_path(row.get("image_path_absolute"))

    # Cache-based paths
    cache_root = row.get("image_cache_root")
    asset = row.get("cauldron_image_asset")
    if isinstance(cache_root, str) and isinstance(asset, str):
        cache_path = Path(cache_root) / asset
        candidates.append(cache_path)

        # "Normalized" variant used in some older paths
        normalized = Path(
            str(cache_path).replace(
                "code_base/which_vlm/dataset_builder/dataset",
                "dataset",
            )
        )
        if normalized != cache_path:
            candidates.append(normalized)

    return candidates


In [46]:
# %%
def pick_first_existing_path(paths: Iterable[Path]) -> Optional[Path]:
    """Return the first path that exists, else None."""
    for p in paths:
        if p.exists():
            return p
    return None


In [47]:
# %%
def fetch_and_cache_image(
    *,
    source_config: Optional[str],
    source_index: Optional[int],
    image_hash: Optional[str],
    image_root: Path,
    fetch_fn=None,
    destination: Optional[Path] = None,
    prefer_local_cache: bool = True,
) -> Optional[Path]:
    """
    Retrieve a Cauldron image and cache it locally.

    If `fetch_fn` is None, we only succeed if the file already exists.
    """
    if source_config is None or source_index is None:
        logger.debug("fetch_and_cache_image: missing source_config or source_index")
        return None

    cache_dir = image_root / source_config
    cache_dir.mkdir(parents=True, exist_ok=True)

    filename = image_hash if image_hash else f"{int(source_index)}"
    target_path = destination or (cache_dir / f"{filename}.png").resolve()

    # Fast path: already cached
    if target_path.exists():
        logger.debug("Image already cached at %s", target_path)
        return target_path

    if fetch_fn is None:
        logger.warning(
            "fetch_and_cache_image: no fetch_fn and file does not exist: %s",
            target_path,
        )
        return None

    try:
        image, _ = fetch_fn(
            source_config=source_config,
            source_index=int(source_index),
            image_hash=image_hash,
            prefer_local_cache=prefer_local_cache,
            image_root=image_root,
        )
    except Exception as exc:
        logger.warning(
            "fetch_and_cache_image: failed to fetch %s/%s: %s",
            source_config,
            source_index,
            exc,
        )
        return None

    try:
        image.save(target_path)
        logger.debug("Saved image to %s", target_path)
    except Exception as exc:
        logger.warning("fetch_and_cache_image: failed to save %s: %s", target_path, exc)
        return None

    return target_path


In [48]:
# %%
def build_cauldron_lookup(
    table_path: Path,
    image_root: Path,
    *,
    fetch_fn=None,
    eager_download: bool = False,
    max_rows: Optional[int] = None,
) -> Dict[str, Path]:
    """
    Map `dataset:index` lookup keys to image paths.

    If `eager_download=True` and `fetch_fn` is provided, we try to download
    missing images immediately. Otherwise we only index existing files.
    """
    if not table_path.exists():
        logger.warning("build_cauldron_lookup: table_path not found: %s", table_path)
        return {}

    df = pd.read_parquet(
        table_path,
        columns=["source_dataset", "source_index", "image_path"],
    )
    if max_rows is not None:
        df = df.head(max_rows)

    df = df.assign(
        dataset_name=df["source_dataset"].str.replace(
            "the_cauldron_", "", regex=False
        ),
        lookup_key=lambda frame: frame.apply(
            lambda r: build_lookup_key(r["dataset_name"], r["source_index"]),
            axis=1,
        ),
        filename=lambda frame: frame["image_path"].apply(lambda p: Path(p).name),
    )

    df["resolved_path"] = (
        image_root / df["dataset_name"] / df["filename"]
    ).apply(lambda p: Path(p).resolve())

    exists_mask = df["resolved_path"].apply(lambda p: p.exists())
    existing_df = df[exists_mask].copy()

    lookup: Dict[str, Path] = dict(
        zip(existing_df["lookup_key"], existing_df["resolved_path"])
    )

    logger.info(
        "build_cauldron_lookup: %d/%d images already exist on disk",
        exists_mask.sum(),
        len(df),
    )

    if eager_download and fetch_fn is not None:
        missing_df = df[~exists_mask].copy()
        logger.info(
            "build_cauldron_lookup: attempting downloads for %d missing images",
            len(missing_df),
        )

        for row in tqdm(
            missing_df.itertuples(index=False),
            total=len(missing_df),
            desc="Downloading Cauldron images",
        ):
            dataset_name = row.dataset_name
            source_index = int(row.source_index)
            lookup_key = row.lookup_key
            target_path = Path(row.resolved_path)

            downloaded = fetch_and_cache_image(
                source_config=dataset_name,
                source_index=source_index,
                image_hash=None,
                image_root=image_root,
                fetch_fn=fetch_fn,
                destination=target_path,
                prefer_local_cache=False,
            )
            if downloaded is not None:
                lookup[lookup_key] = downloaded

    logger.info("build_cauldron_lookup: final lookup size = %d", len(lookup))
    return lookup


In [49]:
# %%
def resolve_row_image_path(
    row: pd.Series,
    lookup_map: Dict[str, Path],
    *,
    project_root: Path,
    image_root: Path,
    fetch_fn=None,
) -> Optional[Path]:
    """
    Resolve the best on-disk image path for this row.

    Order:
    1. cauldron_lookup_key → lookup_map
    2. local candidate paths in the row
    3. optional fetch via fetch_fn
    """
    # 1) lookup_map
    lookup_key = row.get("cauldron_lookup_key")
    if isinstance(lookup_key, str) and lookup_key in lookup_map:
        return lookup_map[lookup_key]

    # 2) candidate paths
    candidates = candidate_paths_from_row(row, project_root=project_root)
    existing = pick_first_existing_path(candidates)
    if existing is not None:
        return existing

    # 3) fetch
    if fetch_fn is not None:
        source_config = (
            row.get("source_config")
            or row.get("config")
            or row.get("source_dataset")
        )
        source_index = row.get("source_index")
        image_hash = row.get("image_bytes_hash")

        if source_config is not None and source_index is not None:
            downloaded = fetch_and_cache_image(
                source_config=str(source_config),
                source_index=int(source_index),
                image_hash=image_hash if isinstance(image_hash, str) else None,
                image_root=image_root,
                fetch_fn=fetch_fn,
                prefer_local_cache=True,
            )
            if downloaded is not None:
                return downloaded

    return None


In [50]:
DATASET_PATH

PosixPath('/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/dataset/final_dataset/router_pivot_dataset_train.parquet')

In [51]:
# %% [markdown]
# ## 4. Load router dataset & basic EDA

# %%
if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATASET_PATH}")

df = pd.read_parquet(DATASET_PATH)
print("Rows:", len(df))
print("Columns:", df.columns.tolist())

df.head()


Rows: 37504
Columns: ['sample_id', 'run_id', 'timestamp_utc', 'image_path', 'image_bytes_hash', 'prompt_raw', 'prompt_formatted', 'system_prompt', 'source_dataset', 'source_config', 'router_task', 'ground_truth', 'ground_truth_type', 'mc_options', 'source_index', 'img_width', 'img_height', 'img_aspect_ratio', 'img_file_size_bytes', 'txt_prompt_length_chars', 'txt_prompt_length_words', 'txt_question_type', 'txt_has_mc_options', 'cauldron_image_asset', 'cauldron_lookup_key', 'image_cache_root', 'n_models', 'deepseek_ocr__model_name', 'deepseek_ocr__model_id', 'deepseek_ocr__response_raw', 'deepseek_ocr__response_parsed', 'deepseek_ocr__response_length_chars', 'deepseek_ocr__response_length_tokens', 'deepseek_ocr__stop_reason', 'deepseek_ocr__error_message', 'deepseek_ocr__is_refusal', 'deepseek_ocr__ok', 'deepseek_ocr__score_exact_match', 'deepseek_ocr__score_exact_match_normalized', 'deepseek_ocr__score_contains_gt', 'deepseek_ocr__score_gt_in_response', 'deepseek_ocr__score_f1', 'deeps

,sample_id,run_id,timestamp_utc,image_path,image_bytes_hash,prompt_raw,prompt_formatted,system_prompt,source_dataset,source_config,...,gemma_3_27b__semantic_f1_f1,gemma_3_27b__semantic_f1_gen_statements,gemma_3_27b__semantic_f1_gt_statements,gemma_3_27b__semantic_f1_matches,gemma_3_27b__semantic_f1_labels,gemma_3_27b__glider_score,gemma_3_27b__glider_reasoning,gemma_3_27b__glider_highlight,gemma_3_27b__glider_raw_output,subset_split
0,ai2d_00000_45f9e7163ea99b4c,exp_20251127_132944,2025-11-27T18:30:57.474332,None,45f9e7163ea99b4c,Question: What do respiration and combustion g...,None,None,cauldron_ai2d,ai2d,...,None,None,None,None,None,4,- The model's output is semantically correct a...,"[B, carbon dioxide, respiration, combustion, c...",<reasoning>\n- The model's output is semantic...,train
1,ai2d_00001_0135592f21ea024d,exp_20251127_132944,2025-11-27T18:31:03.209660,None,0135592f21ea024d,"Question: From the given food web, name any tw...",None,None,cauldron_ai2d,ai2d,...,None,None,None,None,None,4,- The model's answer is semantically correct a...,"[Jack Rabbit, Jack Rabbit, herbivores, eats pl...",<reasoning>\n- The model's answer is semantic...,train
2,ai2d_00003_49e0ce3c07c66e5c,exp_20251127_132944,2025-11-27T18:31:15.901068,None,49e0ce3c07c66e5c,Question: What process does this diagram portr...,None,None,cauldron_ai2d,ai2d,...,None,None,None,None,None,5,- The model's output correctly identifies the ...,"[Photosynthesis, plant, sunlight, carbon dioxi...",<reasoning>\n- The model's output correctly i...,train
3,ai2d_00005_a70e661f5c7ae68d,exp_20251127_132944,2025-11-27T18:31:27.716607,None,a70e661f5c7ae68d,Question: Which type of rock consists of molte...,None,None,cauldron_ai2d,ai2d,...,None,None,None,None,None,5,"- The model's answer is exactly correct, as it...","[Igneous Rocks, molten rock, volcano, magma, l...",<reasoning>\n- The model's answer is exactly ...,train
4,ai2d_00006_e6969306f822b11d,exp_20251127_132944,2025-11-27T18:31:33.353756,None,e6969306f822b11d,Question: What is formed by the deposition of ...,None,None,cauldron_ai2d,ai2d,...,None,None,None,None,None,4,- The model's answer is semantically correct a...,"[Sedimentary, deposition, weathered, remains, ...",<reasoning>\n- The model's answer is semantic...,train


In [52]:
# %%
# Schema & dtypes
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37504 entries, 0 to 37503
Columns: 223 entries, sample_id to subset_split
dtypes: bool(16), float64(56), int64(41), object(110)
memory usage: 59.8+ MB


In [53]:
# %%
# Basic null counts for key columns
key_cols = [
    "sample_id",
    "prompt_formatted",
    "prompt_raw",
    "router_task",
    "ground_truth",
    "ground_truth_type",
    "image_path",
    "image_cache_root",
    "cauldron_lookup_key",
]
for col in key_cols:
    if col in df.columns:
        print(f"{col}: nulls = {df[col].isna().sum()} / {len(df)}")


sample_id: nulls = 0 / 37504
prompt_formatted: nulls = 37504 / 37504
prompt_raw: nulls = 0 / 37504
router_task: nulls = 0 / 37504
ground_truth: nulls = 0 / 37504
ground_truth_type: nulls = 0 / 37504
image_path: nulls = 37504 / 37504
image_cache_root: nulls = 0 / 37504
cauldron_lookup_key: nulls = 0 / 37504


In [54]:
# %%
# Quick distributions
if "router_task" in df.columns:
    print("router_task value_counts:")
    display(df["router_task"].value_counts(dropna=False).head(20))

if "ground_truth_type" in df.columns:
    print("\nground_truth_type value_counts:")
    display(df["ground_truth_type"].value_counts(dropna=False))


router_task value_counts:


router_task
chart_reasoning        5525
document_ocr           4187
table_reasoning        2801
spatial_reasoning      2796
geometry_reasoning     2306
general_vqa            1426
dense_captioning       1425
rendered_text_ocr      1423
knowledge_vqa          1418
meme_classification    1416
diagram_reasoning      1415
icon_reasoning         1404
abstract_reasoning     1404
chart_captioning       1401
map_reasoning          1399
medical_report         1397
code_generation        1392
table_math             1381
handwriting_ocr        1378
diagram_captioning      210
Name: count, dtype: int64


ground_truth_type value_counts:


ground_truth_type
exact       26075
freeform     5825
mc           4223
numeric      1381
Name: count, dtype: int64

In [55]:
# %% [markdown]
# ## 5. Build Cauldron lookup & resolve image paths

# %%
lookup_map = build_cauldron_lookup(
    CAULDRON_LOOKUP_PATH,
    IMAGE_ROOT,
    fetch_fn=fetch_cauldron_image,
    eager_download=False,   # set True if you want to pre-download
    max_rows=None,          # or a small number for debugging
)

df_unique = df.drop_duplicates(subset=["sample_id"]).copy()
print("Unique sample_ids:", len(df_unique))

df_unique["resolved_image_path"] = df_unique.apply(
    lambda row: resolve_row_image_path(
        row,
        lookup_map,
        project_root=PROJECT_ROOT,
        image_root=IMAGE_ROOT,
        fetch_fn=fetch_cauldron_image,
    ),
    axis=1,
)

num_with_image = df_unique["resolved_image_path"].notna().sum()
print(f"Samples with resolved_image_path: {num_with_image} / {len(df_unique)}")

df_unique[["sample_id", "router_task", "resolved_image_path"]].head(10)


INFO:vlm_router:build_cauldron_lookup: 256/256 images already exist on disk
INFO:vlm_router:build_cauldron_lookup: final lookup size = 256


Unique sample_ids: 37504


INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/HuggingFaceM4/the_cauldron/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/HuggingFaceM4/the_cauldron/847a98a779b1652d65111daf20c972dfcd333605/README.md "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/HuggingFaceM4/the_cauldron/resolve/847a98a779b1652d65111daf20c972dfcd333605/the_cauldron.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/HuggingFaceM4/the_cauldron/HuggingFaceM4/the_cauldron.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/HuggingFaceM4/the_cauldron/revision/847a98a779b1652d65111daf20c972dfcd333605 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/HuggingFaceM4/the_cauldron/resolve/847a98a779b1652d65111daf20c972dfcd333605/.huggingface.yaml "HTT

KeyboardInterrupt: 

## 6. Display Sample Images

Let's visualize a few samples with their tasks and prompts.

In [ ]:
# Display a few sample images
import matplotlib.pyplot as plt

sample_df = df_unique[df_unique["resolved_image_path"].notna()].head(6)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, (_, row) in enumerate(sample_df.iterrows()):
    if idx >= 6:
        break
    
    img_path = Path(row["resolved_image_path"])
    if img_path.exists():
        img = Image.open(img_path)
        axes[idx].imshow(img)
        axes[idx].axis('off')
        title = f"Task: {row['router_task']}\nID: {row['sample_id'][:12]}..."
        axes[idx].set_title(title, fontsize=10)

plt.tight_layout()
plt.show()

# Display prompts for these samples
print("\nSample prompts:")
for idx, (_, row) in enumerate(sample_df.head(3).iterrows()):
    print(f"\n{'='*80}")
    print(f"Sample {idx+1}: {row['router_task']}")
    print(f"{'='*80}")
    prompt = row.get('prompt_raw', '')
    print(f"Prompt: {prompt[:200]}..." if len(str(prompt)) > 200 else f"Prompt: {prompt}")

## 7. Configure CascadeFlow Experiment

Now we'll use the `cascadeflow_experiment.py` module to run the routing experiment.

In [ ]:
# Import the experiment module
import cascadeflow_experiment as cf_exp

# Check if module loaded correctly
print("Available functions:", [x for x in dir(cf_exp) if not x.startswith('_')])

In [ ]:
# Configure CascadeFlow experiment
# IMPORTANT: Adjust these URLs to match your running vLLM servers!

model_specs = [
    cf_exp.ModelSpec(
        name="qwen2-vl-2b-instruct",
        base_url="http://localhost:8000/v1",
        cost=0.00002,  # Cheapest - OCR specialist
        temperature=0.1,
        speed_ms=300,
        keywords=["vision", "multimodal", "ocr"],
        domains=["vlm"],
    ),
    cf_exp.ModelSpec(
        name="qwen2.5-vl-3b-instruct",
        base_url="http://localhost:8001/v1",
        cost=0.00004,  # Best cost-performance balance
        temperature=0.1,
        speed_ms=400,
        keywords=["vision", "multimodal"],
        domains=["vlm"],
    ),
    cf_exp.ModelSpec(
        name="qwen2.5-vl-7b-instruct",
        base_url="http://localhost:8002/v1",
        cost=0.00006,  # Good for complex reasoning
        temperature=0.1,
        speed_ms=600,
        keywords=["vision", "multimodal", "reasoning"],
        domains=["vlm"],
    ),
    cf_exp.ModelSpec(
        name="Qwen2.5-VL-72B-Instruct",
        base_url="http://localhost:8003/v1",
        cost=0.00012,  # Largest - complex tasks
        temperature=0.15,
        speed_ms=1200,
        keywords=["vision", "multimodal", "reasoning", "complex"],
        domains=["vlm"],
    ),
]

# Create experiment config
config = cf_exp.ExperimentConfig(
    dataset_path=DATASET_PATH,
    cauldron_lookup_path=CAULDRON_LOOKUP_PATH,
    image_root=IMAGE_ROOT,
    output_dir=OUTPUT_DIR,
    cascade_models=model_specs,
    project_root=PROJECT_ROOT,
    max_samples=100,  # Start small for testing
    seed=42,
    generation_max_tokens=300,
    generation_temperature=0.1,
    verbose_agent=False,
    experiment_name="cascadeflow_vlm_router",
)

print("Experiment Configuration")
print("="*60)
print(f"Dataset: {config.dataset_path.name}")
print(f"Max samples: {config.max_samples}")
print(f"Seed: {config.seed}")
print(f"\nCascade Models ({len(config.cascade_models)}):")
for i, spec in enumerate(config.cascade_models, 1):
    print(f"  {i}. {spec.name}")
    print(f"     URL: {spec.base_url}")
    print(f"     Cost: ${spec.cost:.6f}/request")
    print(f"     Speed: ~{spec.speed_ms}ms")

## 8. Run Experiment

**Prerequisites:**
- vLLM servers must be running at the configured endpoints
- Models must be loaded and ready to serve

This will route each sample through CascadeFlow and record:
- Which model was selected
- Cost and latency
- Whether cascading occurred
- Routing strategy and reason

In [ ]:
# Run the CascadeFlow experiment
print(" Starting CascadeFlow VLM routing experiment...\n")

try:
    results_df, summary, output_paths = cf_exp.run_and_save(config)
    
    print("\n" + "="*80)
    print(" EXPERIMENT COMPLETE")
    print("="*80)
    print(f"\nProcessed {len(results_df)} samples")
    print(f"\nResults saved to:")
    print(f"   Parquet: {output_paths['parquet'].name}")
    print(f"   CSV: {output_paths['csv'].name}")
    
except KeyboardInterrupt:
    print("\n️  Experiment interrupted by user")
    raise
except Exception as e:
    print(f"\n Error running experiment: {e}")
    print("\n Troubleshooting checklist:")
    print("  □ Are vLLM servers running?")
    print("  □ Are the base_url endpoints correct?")
    print("  □ Are models loaded and ready?")
    print("  □ Is network connectivity available?")
    print("  □ Do you have enough GPU memory?")
    raise

## 9. Results Analysis

Analyze routing decisions, costs, and performance.

In [ ]:
# Display summary statistics
print("="*80)
print("CASCADEFLOW ROUTING SUMMARY")
print("="*80)

successful = results_df[results_df['error'].isna()]
failed = results_df[results_df['error'].notna()]

print(f"\nExecution Summary:")
print(f"  Total samples: {len(results_df)}")
print(f"   Successful: {len(successful)} ({len(successful)/len(results_df)*100:.1f}%)")
print(f"   Failed: {len(failed)} ({len(failed)/len(results_df)*100:.1f}%)")

if len(successful) > 0:
    print(f"\n Cost Metrics:")
    print(f"  Mean cost/sample: ${successful['total_cost'].mean():.6f}")
    print(f"  Median cost/sample: ${successful['total_cost'].median():.6f}")
    print(f"  Total cost: ${successful['total_cost'].sum():.6f}")
    
    print(f"\n Latency Metrics:")
    print(f"  Mean latency: {successful['latency_ms'].mean():.0f} ms")
    print(f"  Median latency: {successful['latency_ms'].median():.0f} ms")
    print(f"  P95 latency: {successful['latency_ms'].quantile(0.95):.0f} ms")

if 'model_mix' in summary:
    print(f"\n Model Distribution:")
    display(summary['model_mix'])

if 'cost_summary' in summary:
    print(f"\n Cost by Model:")
    display(summary['cost_summary'])

if len(failed) > 0:
    print(f"\n️  Failures ({len(failed)} samples):")
    print(failed[['sample_id', 'router_task', 'exception_type', 'error']].head(10))

In [ ]:
# Create visualizations
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
successful = results_df[results_df['error'].isna()]

if len(successful) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # 1. Model usage distribution
    ax = axes[0, 0]
    model_counts = successful['model_used'].value_counts()
    colors = sns.color_palette('husl', len(model_counts))
    model_counts.plot(kind='barh', ax=ax, color=colors)
    ax.set_title('Model Usage Distribution', fontsize=14, fontweight='bold', pad=20)
    ax.set_xlabel('Number of Samples', fontsize=12)
    ax.set_ylabel('Model', fontsize=12)
    for i, v in enumerate(model_counts):
        ax.text(v + 0.5, i, str(v), va='center', fontsize=10)
    
    # 2. Cost distribution
    ax = axes[0, 1]
    successful['total_cost'].hist(bins=30, ax=ax, color='coral', edgecolor='black', alpha=0.7)
    mean_cost = successful['total_cost'].mean()
    median_cost = successful['total_cost'].median()
    ax.axvline(mean_cost, color='red', linestyle='--', linewidth=2, label=f'Mean: ${mean_cost:.6f}')
    ax.axvline(median_cost, color='blue', linestyle='--', linewidth=2, label=f'Median: ${median_cost:.6f}')
    ax.set_title('Cost Distribution per Sample', fontsize=14, fontweight='bold', pad=20)
    ax.set_xlabel('Cost ($)', fontsize=12)
    ax.set_ylabel('Frequency', fontsize=12)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # 3. Latency distribution
    ax = axes[1, 0]
    successful['latency_ms'].hist(bins=30, ax=ax, color='lightgreen', edgecolor='black', alpha=0.7)
    mean_lat = successful['latency_ms'].mean()
    median_lat = successful['latency_ms'].median()
    ax.axvline(mean_lat, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_lat:.0f}ms')
    ax.axvline(median_lat, color='blue', linestyle='--', linewidth=2, label=f'Median: {median_lat:.0f}ms')
    ax.set_title('Latency Distribution', fontsize=14, fontweight='bold', pad=20)
    ax.set_xlabel('Latency (ms)', fontsize=12)
    ax.set_ylabel('Frequency', fontsize=12)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # 4. Cost vs Latency by model
    ax = axes[1, 1]
    for model in successful['model_used'].unique():
        model_data = successful[successful['model_used'] == model]
        ax.scatter(model_data['total_cost'], model_data['latency_ms'], 
                  label=model, alpha=0.6, s=50)
    ax.set_title('Cost vs Latency by Model', fontsize=14, fontweight='bold', pad=20)
    ax.set_xlabel('Cost ($)', fontsize=12)
    ax.set_ylabel('Latency (ms)', fontsize=12)
    ax.legend(fontsize=9, loc='best')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("️  No successful samples to visualize")

In [ ]:
# Task-based routing analysis
successful = results_df[results_df['error'].isna()]

if len(successful) > 0 and 'router_task' in successful.columns:
    print("="*80)
    print("ROUTING BY TASK")
    print("="*80)
    
    # Cross-tabulation
    task_model = pd.crosstab(
        successful['router_task'], 
        successful['model_used'],
        margins=True
    )
    
    print("\nTask → Model Routing Matrix:")
    display(task_model)
    
    # Average cost and latency by task
    task_stats = successful.groupby('router_task').agg({
        'total_cost': ['mean', 'std'],
        'latency_ms': ['mean', 'std'],
        'sample_id': 'count'
    }).round(6)
    task_stats.columns = ['Cost Mean', 'Cost Std', 'Latency Mean', 'Latency Std', 'Count']
    task_stats = task_stats.sort_values('Cost Mean', ascending=False)
    
    print("\nTask Performance Metrics:")
    display(task_stats.head(15))
    
    # Visualize task-model heatmap
    task_model_pct = pd.crosstab(
        successful['router_task'], 
        successful['model_used'],
        normalize='index'
    ) * 100
    
    plt.figure(figsize=(12, 8))
    sns.heatmap(task_model_pct, annot=True, fmt='.1f', cmap='YlOrRd', 
                cbar_kws={'label': 'Percentage (%)'})
    plt.title('Model Selection by Task (%)', fontsize=14, fontweight='bold', pad=20)
    plt.xlabel('Model', fontsize=12)
    plt.ylabel('Task', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

## 10. Export for Router Comparison

Save results in a format compatible with the trained VLM router evaluation.

In [ ]:
# Create comparison dataset for router evaluation
if len(results_df) > 0:
    comparison_df = results_df[[
        'sample_id',
        'router_task',
        'model_used',
        'total_cost',
        'latency_ms',
        'cascaded',
        'routing_strategy',
        'routing_reason',
        'error',
    ]].copy()
    
    comparison_df['method'] = 'cascadeflow'
    comparison_df['timestamp'] = pd.Timestamp.now()
    
    # Save to output directory
    comparison_path = OUTPUT_DIR / "cascadeflow_routing_comparison.parquet"
    comparison_df.to_parquet(comparison_path, index=False)
    
    print("="*80)
    print(" EXPORT COMPLETE")
    print("="*80)
    print(f"\nComparison data saved to:")
    print(f"   {comparison_path}")
    print(f"\nDataset info:")
    print(f"  Rows: {len(comparison_df):,}")
    print(f"  Columns: {len(comparison_df.columns)}")
    print(f"  Successful: {comparison_df['error'].isna().sum():,}")
    print(f"  Failed: {comparison_df['error'].notna().sum():,}")
    
    print("\nPreview:")
    display(comparison_df.head(10))
    
    print("\n Next steps:")
    print("  1. Train the VLM router using train_router.py")
    print("  2. Run router inference on the same samples")
    print("  3. Compare routing decisions:")
    print("     - Model selection agreement")
    print("     - Cost efficiency")
    print("     - Latency comparison")
    print("     - Quality/accuracy metrics")
else:
    print("️  No results to export")